In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ---- style (same as trainset_distribution_acrossmodel) ----
def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base - 1,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=20)

colors = {"panav": "#61AFEF", "lacs": "#E06C75"}
labels = {"panav": "rBMRB-PANAV", "lacs": "rBMRB-LACS"}

# ---- data paths (read only) ----
OFFSET_JSON = {
    "panav": "../../data/rBMRB/bmrb_cs_data_withoffset_panav_v2.json",
    "lacs":  "../../data/rBMRB/bmrb_cs_data_withoffset_lacs_v2.json",
}
SPLIT_DIR = "../../data/rBMRB/cluster_30p/PANAV"
ATOM = "N"

METHODS = ["panav", "lacs"]
SPLITS = ["train", "val", "test"]
SPLITS_NUM = ["TRAIN (n=4647)", "VAL (n=385)", "TEST (n=413)"]

def load_data():
    """In each method's (panav/lacs) offset json, 'offset' is {ref_key: {'N': .., 'CA': .., ...}}.
    ref_key is an internal index that does not match the split files, so the join key is pdb_id
    (entry_ID without the 'bmr' prefix, lower-cased). The two sides format the field differently:
    the offset json uses the 'bmr' prefix (e.g. 'bmr19714_1') while the split files do not
    (e.g. '19714_1'); both are normalised to the same format before joining.
    """
    offset_by_pdbid = {}
    for m in METHODS:
        d = json.load(open(OFFSET_JSON[m]))
        pdb2offset = {}
        for key, off in d["offset"].items():
            pdb_id = d["entry_ID"][key].lower().removeprefix("bmr")
            n_off = off.get(ATOM)
            if n_off is not None:
                pdb2offset[pdb_id] = float(n_off)
        offset_by_pdbid[m] = pdb2offset

    data = {sp: {m: [] for m in METHODS} for sp in SPLITS}
    for sp in SPLITS:
        d = json.load(open(f"{SPLIT_DIR}/{sp}.json"))
        pdb_ids = [v.lower().removeprefix("bmr") for v in d["entry_ID"].values()]
        for pdb_id in pdb_ids:
            for m in METHODS:
                v = offset_by_pdbid[m].get(pdb_id)
                if v is not None:
                    data[sp][m].append(v)
    for sp in SPLITS:
        for m in METHODS:
            data[sp][m] = np.asarray(data[sp][m], float)
    return data


# ---------- plot: method x split violins ----------
data = load_data()

fig, ax = plt.subplots(figsize=(9, 6))

centers = {"train": 1.0, "val": 2.0, "test": 3.0}
dx = {"panav": -0.22, "lacs": +0.22}
width = 0.38

for m in METHODS:
    pos = [centers[sp] + dx[m] for sp in SPLITS]
    sets = [data[sp][m] for sp in SPLITS]
    vp = ax.violinplot(sets, positions=pos, widths=width, showmedians=True)
    for body in vp["bodies"]:
        body.set_facecolor(colors[m])
        body.set_edgecolor("black")
        body.set_alpha(0.7)
        body.set_linewidth(0.8)
    for part in ("cmedians", "cbars", "cmins", "cmaxes"):
        if part in vp:
            vp[part].set_edgecolor("black")
            vp[part].set_linewidth(1.0)

ax.axhline(0, color="grey", lw=1.5, linestyle=":", alpha=0.5)
# ax.axhline(0.6, color="grey", lw=1.5, linestyle="--", alpha=0.5)
# ax.axhline(-0.6, color="grey", lw=1.5, linestyle="--", alpha=0.5)
ax.set_xticks(list(centers.values()))
ax.set_xticklabels(SPLITS_NUM)
ax.set_ylabel("$^{15}$N re-referencing offset (ppm)")
ax.set_xlim(0.4, 3.6)
# ax.grid(axis="y", alpha=0.3, linestyle="--")

legend_handles = [Patch(facecolor=colors[m], edgecolor="black", alpha=0.7,
                        label=f"{labels[m]}")
                  for m in METHODS]
for m in METHODS:
    print(f"(μ={np.concatenate([data[sp][m] for sp in SPLITS]).mean():.2f}, "
                              f"σ={np.concatenate([data[sp][m] for sp in SPLITS]).std():.2f})")
# threshold line entry
# legend_handles.append(Line2D([0], [0], color="grey", lw=1, linestyle="--",
#                               label="±0.6 ppm"))
ax.legend(handles=legend_handles, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("Figs/offset_violin_styled.png", dpi=300, bbox_inches="tight")
plt.savefig("Figs/offset_violin_styled.svg", bbox_inches="tight")
plt.show()

for m in METHODS:
    for sp in SPLITS:
        a = data[sp][m]
        print(f"{labels[m]:<6} {sp:<5} n={a.size:>5}  μ={a.mean():.3f}  σ={a.std():.3f}  med={np.median(a):.3f}")